**The conference/seminar has been supported by the Postgraduate Students Conference/Seminar Grants of the Research Grants Council, Hong Kong.**

# Step 0 Download the datatset & Extract Dog-Related Captions


In [ ]:
import pandas as pd
import requests
from io import BytesIO
import zipfile
import json

## Download the Flickr30k zip file
url = "https://cs.stanford.edu/people/karpathy/deepimagesent/flickr30k.zip"
response = requests.get(url)

## Extract the dataset.json file from the zip archive
with zipfile.ZipFile(BytesIO(response.content)) as z:
    with z.open('flickr30k/dataset.json') as f:
        data = json.load(f)

## Parse the JSON and filter captions containing "dog"
captions = []
for item in data['images']:
    for sentence in item['sentences']:
        if 'dog' in sentence['raw'].lower():
            captions.append(sentence['raw'])

## Save filtered captions to CSV file
pd.DataFrame(captions, columns=['caption']).to_csv('dog_captions.csv', index=False)

# Step 1 Import modules

Note: After running the code below, you should select "Restart session" under the "Runtime" menu, and then rerun the code.

In [ ]:
## English tagger
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
lemmatizer = WordNetLemmatizer()

## Install nltk packages (for tagging)
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('wordnet')
nltk.download('stopwords')
nltk.download('omw-1.4')

## Install spellchecker
!pip install pyspellchecker==0.7.2
from spellchecker import SpellChecker
spell = SpellChecker()

## Glob files matching a specified pattern
import os
import glob
import csv

## Preprocess data
import re
from itertools import combinations

## Create a data frame
import pandas as pd

## Keyword analysis & semantic network analysis
!pip install apyori
!pip install mlxtend --upgrade
import networkx as nx
import operator
from sklearn.feature_extraction.text import TfidfVectorizer
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules
from collections import Counter # frequency analysis

## Visualization for semantic network
import matplotlib.pyplot as plt
import matplotlib.colors as mcl
from matplotlib.colors import LinearSegmentedColormap
import numpy as np
from community import community_louvain

## Ignore warnings
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

## Read the dataset
df = pd.read_csv('dog_captions.csv')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 1.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for apyori: filename=apyori-1.1.2-py3-none-any.whl size=5954 sha256=02a0249d610006f09dfb3dab6034c388f89bc3d9aa82febb58a2cfc9b855c2dd
  Stored in directory: /root/.cache/pip/wheels/7f/49/e3/42c73b19a264de37129fadaa0c52f26cf50e87de08fb9804af
Successfully built apyori
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 84.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 81.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 96.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 94.2 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing ins

# Step 2 Construct a corpus

In [ ]:
# @title 2-1

## Extract information (documents, comments, words) from corpus data

# Create an empty list
descriptions = [] # For dog descriptions
words = [] # For extracted content words

# Create a stopword list
stopwords = nltk.corpus.stopwords.words('english')
stopwords.extend(['dog', 'dogs'])

## Read in each description
with open('dog_captions.csv', mode='r', newline='', encoding='utf-8-sig') as file:
    reader = csv.reader(file)
    next(reader)  # skip header
    for row in reader:
        description = row[0]  # caption in the 1st row
        description = re.sub('[^A-Za-z ]',' ', description)
        description = re.sub(' +', ' ', description)
        description = description.lower()
        tokenized = nltk.word_tokenize(description)
        word_list = []

        for word, pos in nltk.pos_tag(tokenized):
            if pos[0] in ['V']:  # keep verbs
                lemma = lemmatizer.lemmatize(word, 'v')
                if (
                    len(word) > 2
                    and lemma not in stopwords
                    and lemma not in spell.unknown([lemma])
                ):
                    word_list.append(lemma)

        if len(word_list) != 0:
            descriptions.append(description)
            words.append(word_list)

In [ ]:
# @title 2-2

## Create a list containing all words (verbs in this case)
all_content_words = [j for i in words for j in i]
len(all_content_words)

11085

In [ ]:
# @title 2-3

## Count frequency
counter_unigram = Counter(all_content_words)
counter_unigram

Counter({'star': 20,
         'color': 71,
         'play': 859,
         'look': 316,
         'move': 16,
         'fight': 107,
         'sleep': 20,
         'sit': 241,
         'lay': 105,
         'tie': 18,
         'hold': 362,
         'lie': 64,
         'run': 2162,
         'surround': 31,
         'stand': 472,
         'turn': 11,
         'shake': 86,
         'fall': 43,
         'leap': 126,
         'jump': 500,
         'wear': 355,
         'grill': 18,
         'hotdogs': 1,
         'chase': 229,
         'prepare': 12,
         'catch': 399,
         'fly': 28,
         'try': 86,
         'get': 81,
         'make': 33,
         'carry': 246,
         'walk': 571,
         'crouch': 15,
         'attempt': 18,
         'throw': 56,
         'push': 11,
         'rest': 20,
         'open': 24,
         'grass': 1,
         'soak': 4,
         'stick': 22,
         'spray': 7,
         'mouth': 9,
         'watch': 84,
         'float': 9,
         'retrieve': 3

# Step 3 Keyword analysis

In [ ]:
# @title 3-1

## Generate co-occurrence pairs
co_occurrence = []
for words_per_text in words:
    for i in range(len(words_per_text)):
        for j in range(i + 1, len(words_per_text)):  # Avoid self-pairing
            co_occurrence.append((words_per_text[i], words_per_text[j]))
co_occurrence

[('color', 'play'),
 ('sleep', 'sit'),
 ('lay', 'tie'),
 ('lay', 'hold'),
 ('lay', 'sit'),
 ('hold', 'sit'),
 ('run', 'surround'),
 ('stand', 'turn'),
 ('leap', 'fall'),
 ('leap', 'fall'),
 ('wear', 'grill'),
 ('wear', 'hold'),
 ('grill', 'hold'),
 ('wear', 'hotdogs'),
 ('hold', 'grill'),
 ('prepare', 'catch'),
 ('catch', 'fly'),
 ('try', 'catch'),
 ('leap', 'catch'),
 ('carry', 'walk'),
 ('jump', 'catch'),
 ('stand', 'sit'),
 ('attempt', 'catch'),
 ('jump', 'catch'),
 ('hold', 'throw'),
 ('rest', 'open'),
 ('run', 'grass'),
 ('get', 'spray'),
 ('jump', 'catch'),
 ('catch', 'watch'),
 ('stick', 'run'),
 ('hold', 'splash'),
 ('hold', 'fill'),
 ('splash', 'fill'),
 ('run', 'carry'),
 ('create', 'run'),
 ('dry', 'shake'),
 ('wash', 'size'),
 ('wash', 'secure'),
 ('size', 'secure'),
 ('hold', 'soap'),
 ('run', 'bite'),
 ('run', 'play'),
 ('stand', 'roll'),
 ('jump', 'catch'),
 ('jump', 'catch'),
 ('color', 'jump'),
 ('color', 'leap'),
 ('touch', 'look'),
 ('age', 'cook'),
 ('leap', 'surrou

In [ ]:
# @title 3-2

## Create a frequency table for co-occurrence pairs
word_pairs = pd.DataFrame(co_occurrence, columns=["word1", "word2"])
word_pairs["frequency"] = 1
word_frequency_df = word_pairs.groupby(["word1", "word2"], as_index=False).sum()
word_frequency_df

,word1,word2,frequency
0,acquaint,sit,1
1,adopt,size,1
2,adorn,pull,1
3,advertise,stand,1
4,age,bald,1
...,...,...,...
1772,wrap,walk,1
1773,wrestle,cover,1
1774,wrestle,fence,1
1775,wrestle,hug,1


In [ ]:
# @title 3-3

## Filter pairs based on minimum frequency
filtered_pairs = word_frequency_df[word_frequency_df["frequency"] >= 5]
filtered_pairs

,word1,word2,frequency
13,age,wear,5
53,attempt,catch,8
153,carry,cover,5
155,carry,follow,6
162,carry,run,11
...,...,...,...
1742,wear,stand,22
1744,wear,strip,6
1746,wear,take,5
1756,wear,walk,26


In [ ]:
# @title 3-4

## Get keywords
G_pos = nx.Graph()
for _, row in filtered_pairs.iterrows():
    G_pos.add_edge(row["word1"], row["word2"], weight=row["frequency"])

## Calculate centrality measures
dgr = nx.degree_centrality(G_pos)  # Degree centrality

## Sort centrality measures
sorted_dgr = sorted(dgr.items(), key=operator.itemgetter(1), reverse=True)

## Print top keywords based on centrality
print("** Degree Centrality **")
for x in range(min(10, len(sorted_dgr))):  # Print top 10 or fewer
    print(sorted_dgr[x])

** Degree Centrality **
('run', 0.5238095238095237)
('wear', 0.30952380952380953)
('hold', 0.26190476190476186)
('walk', 0.23809523809523808)
('stand', 0.23809523809523808)
('catch', 0.19047619047619047)
('jump', 0.19047619047619047)
('sit', 0.16666666666666666)
('look', 0.14285714285714285)
('play', 0.11904761904761904)


# Step 4 Semantic network analysis

In [ ]:
# @title 4-1

## Build the graph
network = nx.Graph()
for index, row in filtered_pairs.iterrows():
    network.add_edge(row['word1'], row['word2'], frequency=row['frequency'])

## Community detection
communities = community_louvain.best_partition(network)

## Create a DataFrame for communities
communities_df = pd.DataFrame(list(communities.items()), columns=['Node', 'Community'])
community_groups = communities_df.groupby('Community')['Node'].apply(list).reset_index()

## Analyze verb communities
for _, row in community_groups.iterrows():
    comm_id = row['Community']
    nodes = row['Node']
    print(f"\nCommunity {comm_id} ({len(nodes)} verbs):")
    print(f"  Verbs: {', '.join(nodes[:10])}")

In [ ]:
# @title 4-2

## Visualize semantic network
plt.figure(figsize = (150,100)); plt.axis('off')


# Assign colors for communities
colors = ['tab:brown', 'tab:pink', 'tab:gray', 'tab:olive', 'tab:cyan', 'tab:blue', 'tab:orange',
          'tab:green', 'tab:red', 'tab:purple', 'tab:black', 'tab:lightgray']
color_map = [colors[communities[node]] for node in network]


# Set the frequency attribute for each node
for i in list(network.nodes()):
    network.nodes[i]['counter_unigram'] = counter_unigram[i]


# Set size of node as a list of frequency of words
node_size = [4000 * nx.get_node_attributes(network, 'counter_unigram')[v] for v in network]

nx.draw_networkx(network, font_size = 50,
                 node_color = color_map, alpha = 0.8, node_size = node_size,
                 with_labels = True,
                 edge_color ='.5', cmap = plt.cm.Blues)